In [1]:
import mpmath as mp
import numpy as np
from sympy import isprime

mp.mp.dps = 50  # For Δₙ < 10^{-16}
phi = (1 + mp.sqrt(5)) / 2
k_star = mp.mpf('0.3')
alpha = mp.pi / 10  # Wedge threshold (tunable for precision/recall)

def theta_prime(n):
    n = int(n)
    if n < 2: return mp.mpf('inf')  # Non-primes
    n_mp = mp.mpf(n)
    mod_term = mp.fmod(n_mp, phi)
    ratio = mod_term / phi
    if ratio <= mp.mpf('1e-50'): raise ValueError("Near-zero ratio")
    return phi * mp.power(ratio, k_star)

def vortex_sieve(n):
    """No-compute sieve: True if in wedge (prime candidate)."""
    theta = theta_prime(n)
    return theta < alpha  # Geometric check only

# Validation: Accuracy at N=100000
N = 100000
primes = [p for p in range(2, N+1) if isprime(p)]
candidates = [n for n in range(2, N+1) if vortex_sieve(n)]
true_pos = len(set(primes) & set(candidates))
precision = true_pos / len(candidates) if candidates else 0
recall = true_pos / len(primes) if primes else 0
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

# Density enhancement in wedge (bins=50)
thetas_p = np.array([float(theta_prime(p)) for p in primes])
hist, _ = np.histogram(thetas_p, bins=50, range=(0, float(mp.pi)))
overall_density = len(primes) / N
max_bin_density = max(hist) / (N / 50)
enhancement = (max_bin_density / overall_density - 1) * 100
print(f"Wedge enhancement: {enhancement:.1f}%")

Precision: 0.0991, Recall: 0.0044
Wedge enhancement: 488.0%


In [2]:
import numpy as np
import math

def sieve_erato(n):
    if n < 2:
        return []
    sieve = np.ones(n + 1, dtype=bool)
    sieve[0:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if sieve[i]:
            sieve[i*i::i] = False
    return np.nonzero(sieve)[0].tolist()

phi = (1 + math.sqrt(5)) / 2
alpha = math.pi / 10

def theta_prime_vec(ns, k_star):
    ns = np.asarray(ns)
    mod_terms = np.fmod(ns, phi)
    ratios = mod_terms / phi
    ratios[ratios < 1e-50] = 1e-50  # Guard underflow
    return phi * (ratios ** k_star)

# Precompute for all n (vectorized)
N = 100000
ns = np.arange(2, N + 1)
primes = sieve_erato(N)
overall_density = len(primes) / N
is_prime_arr = np.zeros(N + 1, dtype=bool)
is_prime_arr[primes] = True

# Sweep k in [0.05, 0.5], step 0.01
best_k, best_enh = 0.3, 0
for k in np.arange(0.05, 0.51, 0.01):
    thetas = theta_prime_vec(ns, k)
    in_wedge = thetas < alpha
    candidates = ns[in_wedge]
    true_pos = np.sum(is_prime_arr[candidates])
    precision = true_pos / len(candidates) if len(candidates) > 0 else 0
    enh = (precision / overall_density - 1) * 100 if overall_density > 0 else 0
    if enh > best_enh:
        best_enh = enh
        best_k = k

print(f"Optimal k*: {best_k:.2f}, Max wedge enhancement: {best_enh:.1f}%")

Optimal k*: 0.15, Max wedge enhancement: 421.3%


In [4]:
import numpy as np
import math
import time
import gc
import warnings

def sieve_erato(n):
    if n < 2:
        return np.array([], dtype=int)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[0:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if sieve[i]:
            sieve[i*i::i] = False
    return np.nonzero(sieve)[0]

phi = (1 + math.sqrt(5)) / 2
alpha = math.pi / 10

def theta_prime_vec(ns, k_star):
    ns = np.asarray(ns)
    mod_terms = np.fmod(ns, phi)
    ratios = mod_terms / phi
    ratios[ratios < 1e-50] = 1e-50
    return phi * (ratios ** k_star)

def compute_enh(sample_indices, ns, is_prime_arr, k, alpha):
    try:
        sampled_ns = ns[sample_indices]
        res_density = np.sum(is_prime_arr[sampled_ns]) / len(sampled_ns) if len(sampled_ns) > 0 else 0
        thetas = theta_prime_vec(sampled_ns, k)
        del sampled_ns
        gc.collect()
        in_wedge = thetas < alpha
        del thetas
        gc.collect()
        candidates = ns[sample_indices][in_wedge]
        true_pos = np.sum(is_prime_arr[candidates])
        precision = true_pos / len(candidates) if len(candidates) > 0 else 0
        enh = (precision / res_density - 1) * 100 if res_density > 0 else np.nan
        return enh
    except MemoryError:
        warnings.warn("MemoryError in compute_enh; returning NaN")
        return np.nan

# Setup
N = 1000000
ns = np.arange(2, N + 1, dtype=np.int32)
primes = sieve_erato(N)
overall_density = len(primes) / N
is_prime_arr = np.zeros(N + 1, dtype=np.uint8)
is_prime_arr[primes] = 1
num_indices = len(ns)
optimal_k = 0.15
n_resamples = 100
batch_size = 10

# Batched manual bootstrap
start_time = time.time()
enh_samples = []
rng = np.random.default_rng(42)
for batch in range(n_resamples // batch_size):
    batch_enh = []
    for i in range(batch_size):
        try:
            sample_indices = rng.choice(num_indices, size=num_indices, replace=True)
            enh = compute_enh(sample_indices, ns, is_prime_arr, optimal_k, alpha)
            batch_enh.append(enh)
            del sample_indices
            gc.collect()
        except MemoryError:
            warnings.warn(f"MemoryError at resample {(batch * batch_size) + i + 1}; skipping")
    enh_samples.extend(batch_enh)
    gc.collect()

remainder = n_resamples % batch_size
for i in range(remainder):
    sample_indices = rng.choice(num_indices, size=num_indices, replace=True)
    enh = compute_enh(sample_indices, ns, is_prime_arr, optimal_k, alpha)
    enh_samples.append(enh)
    del sample_indices
    gc.collect()

# CI (filter NaN)
enh_samples = np.array(enh_samples)
valid_enh = enh_samples[~np.isnan(enh_samples)]
if len(valid_enh) > 0:
    valid_enh.sort()
    enh_ci_low = np.percentile(valid_enh, 2.5)
    enh_ci_high = np.percentile(valid_enh, 97.5)
else:
    enh_ci_low, enh_ci_high = np.nan, np.nan

exec_time = time.time() - start_time

point_enh = compute_enh(np.arange(num_indices), ns, is_prime_arr, optimal_k, alpha)
print(f"Point Enhancement: {point_enh:.1f}%")
print(f"95% CI: [{enh_ci_low:.1f}, {enh_ci_high:.1f}]% (Completed {len(valid_enh)} valid resamples)")
print(f"Execution Time: {exec_time:.2f}s")

Point Enhancement: 41.5%
95% CI: [-100.0, 262.8]% (Completed 100 valid resamples)
Execution Time: 11.86s
